# Binary search, step by step

Companion to [W10 §2](../README.md#2-binary-search-halve-what-is-left). You will:

1. read the twelve lines of `find()`;
2. watch it halve its window until one item is left;
3. time it against looking at every item;
4. use the same halving on a yes/no question, to split an arch into ribs of equal length.

**Setup.** Run this cell and move on. It lets the notebook find the drawing helpers
(`viz`) and this week's algorithms (`examples/`), whether it sits in the lecture
folder or is your copy in `MyWork/`.

In [ ]:
import sys
import pathlib

HERE = pathlib.Path.cwd()
SHARED = next(p / "Lecture" / "shared" for p in [HERE, *HERE.parents] if (p / "Lecture" / "shared" / "viz.py").exists())
sys.path[:0] = [str(SHARED), str(SHARED.parent / "Lecture_10" / "examples")]

## Step 1 — The whole function

`left` and `right` mark the part of the list where the target can still be. Each pass
of the loop looks at the middle item (`mid`) and keeps only the half that can hold the
target. The list must be sorted, or "the target can only be right of mid" is false.

In [ ]:
import inspect

import binary_search

print(inspect.getsource(binary_search.find))

## Step 2 — Watch the window shrink

Search for 71 in the numbers 0 to 99. `trace` collects the window before each look.
Read down the `size` column: it halves on every line.

In [ ]:
items = list(range(100))
trace = []
index = binary_search.find(items, 71, trace)

print("look   window     size   looked at")
for k, (left, right) in enumerate(trace, start=1):
    print(f"{k:>4}   {left:>2} .. {right:<2}   {right - left + 1:>4}   {items[(left + right) // 2]}")
print("found at index", index)

The same as a picture: one row per look, the blue bar the window, the orange dot the
item looked at.

In [ ]:
from viz import draw_points

rows = [((left, -10 * k), (right, -10 * k)) for k, (left, right) in enumerate(trace)]
looked = [((left + right) // 2, -10 * k) for k, (left, right) in enumerate(trace)]
draw_points(looked, "find 71 in 0..99", colors=["#e8590c"] * len(looked), lines=rows,
            bounds=[[0, 0], [99, -10 * len(trace)]], line_color="#74c0fc", line_width=8)

## Step 3 — Why it matters: time it

Look up 777,777 in a list of a million numbers, once by looking at every item
(`linear_search`), once by halving (`find`). The answer is the same; compare the times.

In [ ]:
import time

million = list(range(1_000_000))
for search in (binary_search.linear_search, binary_search.find):
    start = time.perf_counter()
    found = search(million, 777_777)
    ms = (time.perf_counter() - start) * 1000
    print(f"{search.__name__:<14} index {found}   {ms:8.3f} ms")

## Step 4 — One more look each time the list doubles

The most looks `find` needs is the number of times the list can be halved: log₂ n,
plus one. Doubling the list adds one look. Looking at every item doubles the work.

In [ ]:
import math

print("        n    linear: up to    binary: up to")
for power in (10, 13, 16, 20, 23):
    n = 2 ** power
    print(f"{n:>9,}    {n:>13,}    {math.floor(math.log2(n)) + 1:>13}")

## Step 5 — Bisection: halving a yes/no question

Any question whose answer flips once from no to yes can be halved the same way.
"Is x² at least 2?" is no at x = 0 and yes at x = 2. Each step keeps the half where the
answer flips. After about 30 steps the interval is a billionth wide: x = √2.

In [ ]:
trace = []
root = binary_search.bisect(lambda x: x * x >= 2, 0, 2, trace=trace)
for lo, hi in trace[:6]:
    print(f"between {lo:.6f} and {hi:.6f}")
print(f"... {len(trace)} steps: {root:.9f}   (math.sqrt(2) = {math.sqrt(2):.9f})")

## Step 6 — An arch in ribs of equal length

A parabolic arch, 12 m span, 4 m high, built from 8 straight ribs. Splitting the span
into 8 equal steps in plan gives ribs of different lengths: the steep ones near the
springing are longer. A fabricator wants one rib length.

In [ ]:
span, rise, n = 12.0, 4.0, 8
z = binary_search.parabola(span, rise)

equal_x = [span * k / n for k in range(n + 1)]
print("equal steps in plan, rib lengths:",
      [round(binary_search.arc_length(z, a, b), 3) for a, b in zip(equal_x, equal_x[1:])])

`divide_by_length` asks bisection, for each break k: "is the arc from the springing to
x at least k/8 of the whole arch?" The answer flips once as x grows, so bisection finds
the break. Now every rib has the same length, and the steps in plan are uneven.

In [ ]:
equal_l = binary_search.divide_by_length(z, 0, span, n)
print("equal lengths, rib lengths:      ",
      [round(binary_search.arc_length(z, a, b), 3) for a, b in zip(equal_l, equal_l[1:])])
print("their steps in plan:             ", [round(b - a, 2) for a, b in zip(equal_l, equal_l[1:])])

Both arches as built, side by side: straight ribs between the breaks. Left, the plan
is split evenly; right, the arch is. Compare the first rib with the middle one in each.

In [ ]:
from viz import row

def ribs(xs, title, colour):
    knots = [(x, z(x)) for x in xs]
    return draw_points(knots, title, colors=[colour] * len(knots), lines=list(zip(knots, knots[1:])),
                       bounds=[[0, 0], [span, rise]], size=(460, 260), line_color=colour, line_width=3)


row(ribs(equal_x, "equal steps in plan", "#868e96"), ribs(equal_l, "equal rib lengths", "#1c7ed6"))

## Your turn

1. How many looks does `find` need for a billion items? Work it out with `math.log2`,
   then check with `trace`.
2. Call `find` on an unsorted list, such as `[5, 1, 4, 2, 3]`, for each of its items.
   Which does it miss, and why?
3. Make the arch flatter (`rise = 1`). Do the equal steps in plan come closer to equal
   rib lengths? Why?